# Videos → Gaussian splats (gsplat on Colab)

**Open this notebook from GitHub**, so it's always the latest pushed version:
https://colab.research.google.com/github/Ustavson/loputoo/blob/main/code/video2splat_colab.ipynb

The code (`video2splat.py`, `runs.py`) is pulled from GitHub by the notebook itself. Google Drive holds only the data:
```
MyDrive/Colab Notebooks/loputoo/     ← DRIVE in the first code cell; change it there if yours differs
├── videos/     ← the videos (from data/videos/)
├── prep/       ← frames + COLMAP from the Mac (folders from data/prep/, made by code/prep_local.py)
└── outputs/    ← written by this notebook: one folder per run
```

1. **Runtime → Change runtime type → T4 GPU.**
2. Optionally set `ONLY` in the runs cell (e.g. `"short_1_way_quick"`) to run just some runs.
3. **Runtime → Run all.** Colab warns that the notebook is from GitHub; click *Run anyway*. Keep the tab open; free Colab disconnects idle sessions.

To change the run list, edit `code/runs.py` on the Mac, then commit and push. The notebook picks it up the next time the runs cell runs.

Colab reuses each run's `prep/` folder from Drive. If one is missing, Colab runs COLMAP itself, but that's worse: Colab's 2 CPU cores are slow, COLMAP uses up GPU time while the GPU sits idle, and Colab's COLMAP (3.7 from apt) has no global mapper. On the first iPhone video, the incremental mapper placed only 2 of 200 frames (tested with COLMAP 4.2.1), while the global mapper placed 196.

Per run, `outputs/<run name>/` gets `splat.ply`, `summary.json` (metrics), `renders/` (real vs rendered held-out frames), and the exact settings (`prep.json`, `run.json`, `cfg.yml`). If the session dies, reconnect and **Run all** again; finished runs are skipped.

Rough times on a T4 (not measured yet): install 10–20 min in the first session (compiling CUDA code, saved to `wheels/` in that folder), ~2 min after that. Training 10–20 min per run at `steps=0.25`, about an hour at `steps=1.0`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

# Your project folder on Drive (videos/, prep/; outputs/ and wheels/ are created here).
DRIVE = "/content/drive/MyDrive/Colab Notebooks/loputoo"
import os
assert os.path.isdir(DRIVE), f"{DRIVE} not found: fix DRIVE above"

In [ ]:
# Install. Colab starts every session on a clean machine, so nothing installed survives.
# The slow part is compiling CUDA code (gsplat, fused-ssim, fused-bilagrid). The first session
# compiles them into wheel files and saves them to Drive (DRIVE/wheels/); later
# sessions install the saved wheels in about a minute. The folder name records Python, torch,
# CUDA and GPU type, so if Colab upgrades torch or you get another GPU, it compiles once more.
import glob, os, shutil, subprocess, sys
import torch

GSPLAT = "v1.5.3"
cc = "".join(map(str, torch.cuda.get_device_capability()))   # T4: "75"
key = (f"gsplat-{GSPLAT}_py{sys.version_info.major}{sys.version_info.minor}"
       f"_torch{torch.__version__.split('+')[0]}_cu{torch.version.cuda.replace('.', '')}_sm{cc}")
cache = f"{DRIVE}/wheels/{key}"

def run_cmd(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"failed: {cmd[:300]}")

# COLMAP from apt: only needed when a run has no prep folder on Drive.
run_cmd("apt-get -qq update && apt-get -qq install -y colmap ffmpeg")
# The gsplat repo itself is needed for examples/simple_trainer.py, and to build gsplat.
run_cmd(f"[ -d /content/gsplat ] || git clone -q --branch {GSPLAT} https://github.com/nerfstudio-project/gsplat.git /content/gsplat")
run_cmd("git -C /content/gsplat submodule update -q --init --recursive")

# examples/requirements.txt: the git+ packages are built into wheels (two of them compile CUDA);
# the rest are ordinary PyPI packages, quick to install. numpy<2 is left out: it fights Colab's
# own packages, and the code works on numpy 2.
reqs = [l.strip() for l in open("/content/gsplat/examples/requirements.txt")
        if l.strip() and not l.startswith("#")]
git_reqs = " ".join(f'"{r}"' for r in reqs if r.startswith("git+"))
pypi_reqs = " ".join(f'"{r}"' for r in reqs if not r.startswith(("git+", "numpy")))

if not glob.glob(f"{cache}/*.whl"):
    print(f"No saved wheels for {key}.\nCompiling, first session only: roughly 10-20 min, no output until done...")
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1:]}"   # compile for this GPU type only: much faster
    run_cmd("rm -rf /content/wheels && pip wheel -q --no-build-isolation --no-deps "
            f"-w /content/wheels /content/gsplat {git_reqs}")
    shutil.copytree("/content/wheels", cache + ".partial", dirs_exist_ok=True)   # temp name: a cut-off
    os.rename(cache + ".partial", cache)                                         # copy is never used
    print(f"Saved to {cache}")
try:
    run_cmd(f'pip install -q "{cache}"/*.whl {pypi_reqs}')
except RuntimeError:
    shutil.rmtree(cache)
    raise RuntimeError("Installing the saved wheels failed (output above). Deleted them; "
                       "run this cell again to compile fresh ones.")
compiled = glob.glob("/usr/local/lib/python3*/dist-packages/gsplat/csrc*.so")
print("gsplat CUDA code:", "precompiled" if compiled else "not found, gsplat will compile it on first use (slow)")
print("Installed:", ", ".join(os.path.basename(w).split("-")[0] for w in sorted(glob.glob(f"{cache}/*.whl"))))

In [ ]:
# Code from GitHub (the latest pushed version); data and results on Drive (DRIVE, set in the first cell).
REPO = "/content/loputoo"
ONLY = ""   # run only the runs whose name contains this, e.g. "short_1_way_quick"; "" = all

!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/Ustavson/loputoo.git {REPO}
!git -C {REPO} log -1 --format="code: %h %s (%cr)"

import importlib, os, sys
sys.path.insert(0, f"{REPO}/code")
import runs as runlist
importlib.reload(runlist)
runs = [r for r in runlist.runs() if ONLY in r["name"]]
names = [r["name"] for r in runs]
assert runs, f"no run name contains {ONLY!r}"
for r in runs:
    have = "prep on Drive" if os.path.exists(f"{DRIVE}/prep/{runlist.prep_key(r)}/sparse/0") else "needs COLMAP here"
    print(f"{r['name']:40s} {runlist.prep_key(r):36s} {have}")

In [ ]:
import glob, os, re, shutil, sys
from pathlib import Path
from PIL import Image

missing = sorted({r["video"] for r in runs if not os.path.exists(f"{DRIVE}/videos/{r['video']}")})
assert not missing, f"not found in {DRIVE}/videos/: {missing}"

def sh(args):
    get_ipython().system(" ".join(f'"{a}"' for a in args))

status = {}
for r in runs:
    name = r["name"]
    out = f"{DRIVE}/outputs/{name}"
    if os.path.exists(f"{out}/summary.json"):
        status[name] = "done earlier"
        print(f"=== {name}: already on Drive, skipping")
        continue
    key = runlist.prep_key(r)
    work = f"/content/work/{key}"     # Colab disk: fast, wiped when the session ends
    cache = f"{DRIVE}/prep/{key}"     # frames + COLMAP kept on Drive (from the Mac or an earlier session)
    print(f"\n=== {name}  (frames + COLMAP: {key})")

    # 1. Frames + COLMAP: reuse the Drive copy, or make it here
    if not os.path.exists(work) and os.path.exists(f"{cache}/sparse/0"):
        shutil.copytree(cache, work)
    script = [sys.executable, f"{REPO}/code/video2splat.py", f"{DRIVE}/videos/{r['video']}", work]
    sh(script + runlist.prep_args(r) + ["--stop-after", "sfm"])
    if not os.path.exists(f"{work}/sparse/0/cameras.bin"):
        status[name] = "FAILED in frames / COLMAP (see output above)"
        continue
    if not os.path.exists(cache):   # copy under a temp name so a cut-off copy is never used
        shutil.copytree(work, cache + ".partial", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("database.db", "runs"))
        os.rename(cache + ".partial", cache)

    # 2. Training
    sh(script + runlist.prep_args(r) + ["--gsplat-dir", "/content/gsplat"] + runlist.train_args(r))
    res = f"{work}/runs/{name}"
    if not os.path.exists(f"{res}/summary.json"):
        status[name] = "FAILED in training (see output above)"
        continue

    # 3. Results to Drive (checkpoints stay behind, they are large)
    os.makedirs(out, exist_ok=True)
    shutil.copy(f"{work}/prep.json", out)
    # Renders: gsplat saves a ~4 MB PNG per held-out frame at every evaluation.
    # Keep only the last evaluation, as JPEG (~1 MB each).
    renders = glob.glob(f"{res}/renders/val_step*.png")
    if renders:
        last = max(int(re.search(r"step(\d+)", p).group(1)) for p in renders)
        os.makedirs(f"{out}/renders", exist_ok=True)
        for p in renders:
            if f"step{last}_" in p:
                Image.open(p).convert("RGB").save(f"{out}/renders/{Path(p).stem}.jpg", quality=90)
    for p in ("splat.ply", "summary.json", "run.json", "cfg.yml", "stats"):
        if os.path.isdir(f"{res}/{p}"):
            shutil.copytree(f"{res}/{p}", f"{out}/{p}", dirs_exist_ok=True)
        elif os.path.exists(f"{res}/{p}"):
            shutil.copy(f"{res}/{p}", out)
    status[name] = "ok"

print("\n=== status")
for name, s in status.items():
    print(f"{name:40s} {s}")

In [ ]:
# Compare the runs. Below the table: for each run, one held-out frame (left = real photo, right = render).
import glob, json, re
import pandas as pd
from IPython.display import Image, display

rows = []
for name in names:
    f = f"{DRIVE}/outputs/{name}/summary.json"
    if os.path.exists(f):
        s = json.load(open(f))
        rows.append({"run": name, **{k: s.get(k) for k in (
            "frames", "registered_frames", "trainer_config", "steps_scaler", "trainer_extra_args",
            "psnr", "ssim", "lpips", "num_gaussians", "train_seconds", "ply_mb")}})
display(pd.DataFrame(rows).set_index("run") if rows else "no finished runs yet")

def step_and_index(p):  # val_step6999_0003.jpg -> (6999, -3): latest step, first frame
    step, i = map(int, re.findall(r"\d+", Path(p).name)[:2])
    return step, -i

for name in names:
    renders = glob.glob(f"{DRIVE}/outputs/{name}/renders/val_step*.jpg")
    if renders:
        print(name)
        display(Image(max(renders, key=step_and_index), width=900))

**Viewing:** download `splat.ply` from `outputs/<run name>/` in the Drive folder and drag it into
[SuperSplat](https://superspl.at/editor) in the browser on your Mac.